# Tutorial: Strategy Overrides

Audience:
- Python users who need explicit expert strategy choices.

Prerequisites:
- Familiarity with the intent methods from the earlier notebooks.

Learning goals:
- Import strategies only from `metric.strategies`.
- Override grouping and embedding strategies.
- Recognize roadmap strategy errors.

## Outline

1. Import the public facade.
2. Build a small space.
3. Use promoted strategy overrides.
4. Inspect roadmap-only strategy errors.
5. Try a small exercise.

In [ ]:
from metric import Space
from metric import metrics


## Step 1 - Build the same kind of space

The first code cell intentionally imported no algorithm names.

In [ ]:
records = ["cat", "cot", "coat", "dog", "dogs"]
space = Space(records, metrics.Edit())

assert space.distance(0, 1) == 1
len(space)


## Step 2 - Import strategies in the expert section

Algorithm names live under `metric.strategies`, not at the first level of the tutorial.

In [ ]:
from metric.exceptions import StrategyUnavailableError
from metric.strategies import DiffusionEmbedding, KMedoids, MDS

groups = space.groups(strategy=KMedoids(groups=2))

try:
    space.embed(strategy=MDS(dimensions=2))
    raise AssertionError("embed should require a native binding")
except StrategyUnavailableError as exc:
    embed_error = str(exc)

assert groups.assignments == (0, 0, 0, 1, 1)
assert groups.cluster_sizes == (3, 2)
assert "native C++ binding" in embed_error
groups.to_dict()


## Step 3 - Roadmap strategies fail explicitly

Importable strategy names can be part of stable documentation before their numerical contracts are promoted.

In [ ]:
roadmap_errors = []
for strategy in (DiffusionEmbedding(dimensions=2),):
    try:
        space.embed(strategy=strategy)
    except StrategyUnavailableError as exc:
        roadmap_errors.append(str(exc))

assert len(roadmap_errors) == 1
roadmap_errors


## Exercise

Change `KMedoids(groups=2)` to `KMedoids(groups=3)` and inspect the medoid IDs.

Common pitfall: importing algorithms from compatibility modules in new examples. New expert examples should import strategy names only from `metric.strategies`.

In [ ]:
groups = space.groups(strategy=KMedoids(groups=3))

assert groups.cluster_count == 3
groups.medoids
